# Anchor mixtures, palettes, harmonisation, stability loops (`fig:app-mixtures`, `AnchorChangeA/B`)

Produces `stability_apps/results/*` -- the four-texture mixing / palette figure candidates and the printed-only `AnchorChangeA`/`AnchorChangeB` ratios and 14%/32% re-run-noise floors. No file in the archive holds these numbers.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** `experiments/anchor_mixtures.py` reproduces the mixing/measurement from scratch (unverifiable against any cached file, since none exists -- see that script's docstring). This notebook is the original source.

In [ ]:
# ============================================================
# PHASE 1, CELL 1 — TensorFlow only.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob, random, math, time
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True
from google.colab import drive
drive.mount('/content/drive')
BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
LOOP_OUT = f'{BASE}/stability_apps'; os.makedirs(LOOP_OUT, exist_ok=True)

os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0,'/content/TextureMixer')

if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _ls(name, path, file=None):
        spec = importlib.util.spec_from_file_location(name, path)
        m = importlib.util.module_from_spec(spec); sys.modules[name]=m
        spec.loader.exec_module(m); return m
    def _fm(name, path=None):
        spec = importlib.machinery.PathFinder().find_spec(name, path)
        if spec is None: raise ImportError(name)
        return (None, spec.origin, ('.py','r',1))
    _imp.load_source=_ls; _imp.find_module=_fm; _imp.new_module=lambda n: types.ModuleType(n)
    _imp.reload=importlib.reload; _imp.acquire_lock=lambda: None
    _imp.release_lock=lambda: None; _imp.PY_SOURCE=1
    sys.modules['imp']=_imp

import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1 = _tf2.compat.v1
tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1,'Dimension'): tf1.Dimension = getattr(_tf2,'Dimension',int)
!pip install -q tf_slim
import tf_slim as _slim
_c = types.ModuleType('tensorflow.contrib'); _c.slim=_slim; _c.framework=_slim; _c.layers=_slim
for n in ['tensorflow.contrib','tensorflow.contrib.slim','tensorflow.contrib.framework','tensorflow.contrib.layers']:
    sys.modules[n] = _c if n=='tensorflow.contrib' else _slim
tf1.contrib=_c
sys.modules['tensorflow'] = tf1

_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
open(_p,'w').write(_s.replace('v.shape.ndims is 0','v.shape.ndims == 0')
                     .replace('v.shape.ndims is 1','v.shape.ndims == 1'))

CKPT = f'{TM_STORE}/earth_texture/network-final.pkl'
assert os.path.exists(CKPT), 'checkpoint missing — see the earlier download cell'
import tfutil, misc
sess = tfutil.create_session(force_as_default=True)
nets = {getattr(o,'name',f'_{i}'): o for i,o in enumerate(misc.load_pkl(CKPT))}
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK —', Gs.input_shapes, '->', Gs.output_shapes)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — generate TM degradation loops and stripes.
# ============================================================
import pandas as pd
TM_RES, LATENT_IDX = 128, 0
def tm_prep(p):
    a = np.asarray(p.convert('RGB').resize((TM_RES,)*2), dtype=np.float32)
    return (a/127.5-1.0).transpose(2,0,1)[None]
def tm_pil(a):
    return Image.fromarray(np.clip((a[0].transpose(1,2,0)+1)*127.5,0,255).astype(np.uint8))
def tm_encode(p):
    x = tm_prep(p)
    return (Es_zg.run(x, return_as_list=True)[LATENT_IDX],
            Es_zl.run(x, return_as_list=True)[LATENT_IDX])
def tm_gen(zg, zl):
    return tm_pil(Gs.run(np.tile(zg,(1,1,zl.shape[2],zl.shape[3])), zl, return_as_list=True)[0])
def tm_interp(A, B, e=0.5):
    zgA,zlA = tm_encode(A); zgB,zlB = tm_encode(B)
    return tm_gen((1-e)*zgA+e*zgB, (1-e)*zlA+e*zlB)

DTD = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD) if os.path.isdir(f'{DTD}/{c}')])
assert len(cats) >= 45, f'only {len(cats)} categories'
def safe_open(p, tries=5):
    for i in range(tries):
        try:
            im=Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i==tries-1: raise
            time.sleep(0.4*(i+1))

N_PAIRS, N_LOOP = 20, 8
rng = random.Random(5808)
pairs = []
for i in range(N_PAIRS):
    a,b = rng.sample(cats,2)
    pairs.append((rng.choice(sorted(glob.glob(f'{DTD}/{a}/*.jpg'))),
                  rng.choice(sorted(glob.glob(f'{DTD}/{b}/*.jpg')))))
pd.DataFrame({'k':range(N_PAIRS),'A':[p[0] for p in pairs],'B':[p[1] for p in pairs]}) \
  .to_csv(f'{LOOP_OUT}/pairs.csv', index=False)

t0=time.time()
for k,(pa,pb) in enumerate(pairs):
    A, B = safe_open(pa), safe_open(pb)
    A.resize((TM_RES,)*2).save(f'{LOOP_OUT}/{k:03d}_A.png')
    B.resize((TM_RES,)*2).save(f'{LOOP_OUT}/{k:03d}_B.png')
    x = A
    for t in range(N_LOOP):                       # repeated half-step toward B
        x = tm_interp(x, B, 0.5)
        x.save(f'{LOOP_OUT}/tm_{k:03d}_loop{t:02d}.png')
    for e in [-0.3,-0.15,0.0,0.25,0.5,0.75,1.0,1.15,1.3]:   # incl. extrapolation
        tm_interp(A, B, e).save(f'{LOOP_OUT}/tm_{k:03d}_eta{int(e*100):+04d}.png')
    if k%5==0: print(f'  {k}/{N_PAIRS} ({(time.time()-t0)/60:.1f}m)')
print('\nPHASE 1 DONE. Now: Runtime -> Restart session. Then run cells 3+.')

In [ ]:
# ============================================================
# PHASE 2, CELL 3 — torch only. Run AFTER restarting.
# ============================================================
!pip install -q pot plenoptic einops timm
import os, sys, glob, random, time, math, json
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile, ImageDraw, ImageFont, ImageFilter
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/DATA_ROOT')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
LOOP_OUT = f'{BASE}/stability_apps'
OUT = f'{BASE}/stability_apps/results'; os.makedirs(OUT, exist_ok=True)
TMP='/content/tmp'; os.makedirs(TMP, exist_ok=True)
TMPV='/content/vtmp'; os.makedirs(TMPV, exist_ok=True)

DTD_ROOT = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
assert len(cats) >= 45, f'only {len(cats)} categories — check DTD_ROOT'
def safe_open(p, tries=5):
    for i in range(tries):
        try:
            im=Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i==tries-1: raise
            time.sleep(0.4*(i+1))
def cat_of(p): return Path(p).parent.name
print(f'{len(cats)} categories')

# ---- CNT ----
os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(d):
    a=((d.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(br,bf,si):
    with torch.no_grad():
        L=model.splat(br)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf
        L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms=FMS):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {sz: torch.einsum("bnc,bnhw->bchw",bf,sp[sz]) for sz in set(fms)}
def decode_with_style_maps(br, sm):
    with torch.no_grad():
        L=model.splat(br); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=device)/N; b=torch.ones(M,1,device=device)/M; u=torch.ones(N,1,device=device)
    for _ in range(n): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return (P/(P.sum(1,keepdim=True)+1e-10))@tgt
def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    it=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_ot_sym(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    it=(0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d, eta):
    sa=build_style_maps(d["bf_a"],d["si_a"]); sb=build_style_maps(d["bf_b"],d["si_b"])
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}
def our_interp(m, A, B, eta):
    return to_pil(FNS[m](build_d(cnt_load(A), cnt_load(B)), eta))

# ---- Vacher ----
os.chdir('/content'); os.system('rm -rf texture-interpolation')
os.system('git clone -q --depth 1 https://github.com/JonathanVacher/texture-interpolation.git')
VACHER_DIR='/content/texture-interpolation/texture-synthesis-algorithm'
os.chdir(CNT_DIR)
def prep_square(x, out, size=IMG_SIZE):
    (safe_open(x) if isinstance(x,str) else x).convert('RGB').resize((size,size)).save(out); return out
def vacher_interp(pA, pB, w=0.5, mname='tex_wasser', n_iter=30, bfgs=20, seed=0):
    cwd=os.getcwd(); os.chdir(VACHER_DIR)
    if VACHER_DIR not in sys.path: sys.path.insert(0, VACHER_DIR)
    for m in [m for m in list(sys.modules) if m.startswith(('models','options'))]: del sys.modules[m]
    from options.base_options import BaseOptions
    from models import create_model, utils
    bak=sys.argv
    sys.argv=['run.py','--input_tex_1',str(pA),'--input_tex_2',str(pB),'--weight_1',str(w),
              '--model',mname,'--n_iter',str(n_iter),'--bfgs_iter',str(bfgs),
              '--shape','256,256','--seed',str(seed),'--display_freq','100000']
    try:
        opt=BaseOptions().parse(); m=create_model(opt); m.get_net_and_loss()
        for _ in range(opt.n_iter+1): m.optimize_parameters()
        arr=utils.tensor2im(m.output_tex.data.clamp_(0,1))
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB').resize((IMG_SIZE,)*2)
    finally:
        sys.argv=bak; os.chdir(cwd)

# ---- observers ----
import timm
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
def dino_dist(a,b):
    f=dino_feats([a,b]); return float(np.linalg.norm(f[0]-f[1]))
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
import plenoptic as po
ps_model = po.models.PortillaSimoncelli((IMG_SIZE,IMG_SIZE), n_scales=4, n_orientations=4, spatial_corr_width=7).to(device)
def _raw_ps(p):
    g=p.convert('L').resize((IMG_SIZE,)*2)
    x=torch.from_numpy(np.asarray(g).copy()).float()[None,None].to(device)/255.
    with torch.no_grad(): return ps_model(x).detach().cpu().numpy().ravel()
VALID=~np.isnan(_raw_ps(Image.effect_noise((256,256),60)))
def ps_stats(p): return _raw_ps(p)[VALID]

def manifold(featfn, K=3, n=400, res=IMG_SIZE, seed=0):
    rr=random.Random(seed); imgs=[]
    for i in range(n):
        try: imgs.append(safe_open(rr.choice(sorted(glob.glob(f'{DTD_ROOT}/{rr.choice(cats)}/*.jpg')))).resize((res,)*2))
        except OSError: continue
    Phi=featfn(imgs); Phi=Phi[~np.isnan(Phi).any(axis=1)]
    D=np.linalg.norm(Phi[:,None]-Phi[None,:],axis=2); np.fill_diagonal(D,np.inf)
    rad=np.sort(D,axis=1)[:,K-1]; keep=rad<=np.median(rad)
    Pr,rrad=Phi[keep],rad[keep]
    return lambda f: float(np.max(rrad/(np.linalg.norm(Pr-f,axis=1)+1e-9)))
print('\nPHASE 2 READY')

In [ ]:
# ============================================================
# CELL 4 — degradation loop: x_{t+1} = M(x_t, B, 0.5), starting at A.
# Measures whether repeated interpolation stays on the texture manifold.
# TM frames come from Phase 1; ours and Vacher are generated here.
# ============================================================
pairs = pd.read_csv(f'{LOOP_OUT}/pairs.csv')
N_LOOP = 8
r_dino = manifold(dino_feats, seed=0)
print('manifold built')

METHODS = list(FNS) + ['vacher_wasser', 'texture_mixer']
rows=[]; t0=time.time()
for _, row in pairs.iterrows():
    k = int(row.k)
    A = Image.open(f'{LOOP_OUT}/{k:03d}_A.png').convert('RGB')
    B = Image.open(f'{LOOP_OUT}/{k:03d}_B.png').convert('RGB')
    fB = dino_feats([B])[0]
    for m in METHODS:
        frames=[]
        if m in FNS:
            x = A
            for t in range(N_LOOP):
                x = our_interp(m, x, B, 0.5); frames.append(x)
        elif m == 'vacher_wasser':
            x = A
            for t in range(min(N_LOOP,4)):        # expensive: 4 iterations only
                qa=prep_square(x, f'{TMPV}/la.png'); qb=prep_square(B, f'{TMPV}/lb.png')
                x = vacher_interp(qa, qb, 0.5, 'tex_wasser'); frames.append(x)
        else:
            fs = sorted(glob.glob(f'{LOOP_OUT}/tm_{k:03d}_loop*.png'))
            frames = [Image.open(f).convert('RGB') for f in fs]
        if not frames: continue
        Fm = dino_feats(frames)
        rec = {'pair':k,'method':m,
               'realism_first': r_dino(Fm[0]), 'realism_last': r_dino(Fm[-1]),
               'realism_drop': r_dino(Fm[0]) - r_dino(Fm[-1]),
               'drift_total': float(np.linalg.norm(Fm[-1]-Fm[0])),
               'dist_to_B_first': float(np.linalg.norm(Fm[0]-fB)),
               'dist_to_B_last':  float(np.linalg.norm(Fm[-1]-fB)),
               'n_frames': len(frames)}
        rec['realism_traj'] = json.dumps([r_dino(f) for f in Fm])
        rows.append(rec)
        if m == 'ot_sym' and k < 3:
            for t,f in enumerate(frames): f.save(f'{OUT}/loop_otsym_{k:03d}_{t:02d}.png')
    print(f'  pair {k}/{len(pairs)} ({(time.time()-t0)/60:.1f}m)')

dfl = pd.DataFrame(rows); dfl.to_csv(f'{OUT}/degradation_loop.csv', index=False)
print(f"\n=== DEGRADATION LOOP (n={len(pairs)} pairs) ===")
print(f"  {'method':16} {'realism t=0':>12} {'realism t=N':>12} {'drop':>8} "
      f"{'converged to B':>15}")
for m in METHODS:
    s = dfl[dfl.method==m]
    if not len(s): continue
    conv = (s.dist_to_B_last < s.dist_to_B_first).mean()
    print(f"  {m:16} {s.realism_first.median():12.4f} {s.realism_last.median():12.4f} "
          f"{s.realism_drop.median():8.4f} {100*conv:14.0f}%")
print("\n  drop > 0 = the method degrades under iteration")

fig, ax = plt.subplots(1,2, figsize=(12,4))
for m in METHODS:
    s = dfl[dfl.method==m]
    if not len(s): continue
    tr = np.array([json.loads(t) for t in s.realism_traj if len(json.loads(t))>=4])
    if len(tr): ax[0].plot(range(tr.shape[1]), np.median(tr,axis=0), marker='o', label=m)
ax[0].set_xlabel('loop iteration'); ax[0].set_ylabel('DINOv2 realism'); ax[0].set_title('degradation under iteration')
ax[0].grid(alpha=.25); ax[0].legend(fontsize=8)
ax[1].bar(range(len(METHODS)), [dfl[dfl.method==m].realism_drop.median() if len(dfl[dfl.method==m]) else 0 for m in METHODS])
ax[1].set_xticks(range(len(METHODS))); ax[1].set_xticklabels(METHODS, rotation=30, ha='right', fontsize=8)
ax[1].set_ylabel('realism drop'); ax[1].set_title('total degradation'); ax[1].grid(alpha=.25, axis='y')
plt.tight_layout(); plt.savefig(f'{OUT}/degradation.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# CELL 5 — novelty: is the midpoint just retrieving a source or a
# nearby real texture? NN search in pixel and DINO space.
# ============================================================
N_REF = 300
rr = random.Random(77)
ref_paths = [rr.choice(sorted(glob.glob(f'{DTD_ROOT}/{rr.choice(cats)}/*.jpg'))) for _ in range(N_REF)]
ref_imgs = [safe_open(p).resize((IMG_SIZE,)*2) for p in ref_paths]
REF_D = dino_feats(ref_imgs)
REF_P = np.stack([np.asarray(im, dtype=np.float32).ravel()/255. for im in ref_imgs])
print(f'reference set: {len(ref_imgs)}')

rows=[]
for _, row in pairs.iterrows():
    k=int(row.k)
    A = Image.open(f'{LOOP_OUT}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    B = Image.open(f'{LOOP_OUT}/{k:03d}_B.png').convert('RGB').resize((IMG_SIZE,)*2)
    mids = {m: our_interp(m, A, B, 0.5) for m in FNS}
    tmp = f'{LOOP_OUT}/tm_{k:03d}_eta+050.png'
    if os.path.exists(tmp): mids['texture_mixer'] = Image.open(tmp).convert('RGB').resize((IMG_SIZE,)*2)
    fA, fB = dino_feats([A])[0], dino_feats([B])[0]
    for m, im in mids.items():
        f = dino_feats([im])[0]
        pix = np.asarray(im, dtype=np.float32).ravel()/255.
        d_ref_d = np.linalg.norm(REF_D - f, axis=1)
        d_ref_p = np.linalg.norm(REF_P - pix, axis=1)
        rows.append({'pair':k,'method':m,
                     'nn_dino': float(d_ref_d.min()), 'nn_pixel': float(d_ref_p.min()),
                     'd_to_A': float(np.linalg.norm(f-fA)), 'd_to_B': float(np.linalg.norm(f-fB)),
                     'nn_is_closer_than_sources': bool(d_ref_d.min() < min(np.linalg.norm(f-fA), np.linalg.norm(f-fB)))})
    if k%5==0: print(f'  {k}/{len(pairs)}')

dfn = pd.DataFrame(rows); dfn.to_csv(f'{OUT}/novelty.csv', index=False)
print(f"\n=== NOVELTY (n={len(pairs)}) ===")
print(f"  {'method':16} {'NN dist (DINO)':>15} {'NN dist (pixel)':>16} {'closer to a real':>17}")
for m in dfn.method.unique():
    s = dfn[dfn.method==m]
    print(f"  {m:16} {s.nn_dino.median():15.2f} {s.nn_pixel.median():16.2f} "
          f"{100*s.nn_is_closer_than_sources.mean():16.0f}%")
print("\n  high NN distance = the midpoint is genuinely novel, not retrieved")

In [ ]:
# ============================================================
# APPLICATION 1 — extrapolation. Barycentric weights extend outside
# [0,1]; latent-lerp and optimisation baselines have no such notion.
# ============================================================
ETAS_X = [-0.3,-0.15,0.0,0.25,0.5,0.75,1.0,1.15,1.3]
K_SHOW = 3
r_d = manifold(dino_feats, seed=1)

rows=[]
for k in range(K_SHOW):
    A = Image.open(f'{LOOP_OUT}/{k:03d}_A.png').convert('RGB')
    B = Image.open(f'{LOOP_OUT}/{k:03d}_B.png').convert('RGB')
    d = build_d(cnt_load(A), cnt_load(B))
    strip = {m: [to_pil(FNS[m](d, e)) for e in ETAS_X] for m in ['ot_sym']}
    tm = [Image.open(f'{LOOP_OUT}/tm_{k:03d}_eta{int(e*100):+04d}.png').convert('RGB')
          if os.path.exists(f'{LOOP_OUT}/tm_{k:03d}_eta{int(e*100):+04d}.png') else None for e in ETAS_X]
    fig, ax = plt.subplots(2 if any(t is not None for t in tm) else 1, len(ETAS_X),
                           figsize=(1.9*len(ETAS_X), 4.2))
    ax = np.atleast_2d(ax)
    for j,(e,im) in enumerate(zip(ETAS_X, strip['ot_sym'])):
        ax[0,j].imshow(im); ax[0,j].axis('off')
        ax[0,j].set_title(f'$\\eta$={e}', fontsize=9,
                          color=('crimson' if (e<0 or e>1) else 'black'))
        rows.append({'pair':k,'method':'ot_sym','eta':e,'realism':r_d(dino_feats([im])[0])})
    if ax.shape[0]>1:
        for j,im in enumerate(tm):
            if im is not None:
                ax[1,j].imshow(im); rows.append({'pair':k,'method':'texture_mixer','eta':ETAS_X[j],
                                                 'realism':r_d(dino_feats([im])[0])})
            ax[1,j].axis('off')
        ax[1,0].set_ylabel('TM')
    plt.suptitle(f'pair {k}: red $\\eta$ are extrapolations beyond the endpoints', fontsize=10)
    plt.tight_layout(); plt.savefig(f'{OUT}/extrapolation_{k:03d}.png', dpi=170, bbox_inches='tight'); plt.show()

dfx = pd.DataFrame(rows); dfx.to_csv(f'{OUT}/extrapolation.csv', index=False)
print("\n=== REALISM vs eta (does quality survive extrapolation?) ===")
for m in dfx.method.unique():
    s = dfx[dfx.method==m].groupby('eta').realism.median()
    print(f"  {m:16} " + "  ".join(f"{e:+.2f}:{v:.3f}" for e,v in s.items()))

In [ ]:
# ============================================================
# APPLICATION 2 — k-way barycentre painted back into each region.
# No baseline supports arbitrary-k simplex weights.
# ============================================================
@torch.no_grad()
def bary_style(paths, weights, anchor=0, n_sub=4096):
    flats=[]; ref=None
    for i,p in enumerate(paths):
        blob,bf,si = get_gen_inputs(cnt_load(safe_open(p) if isinstance(p,str) else p))
        sm = build_style_maps(bf,si)[max(FMS)][0]; C,H,W = sm.shape
        flats.append(sm.permute(1,2,0).reshape(-1,C))
        if i==anchor: ref, shape = blob, (C,H,W)
    C,H,W = shape; base = flats[anchor]; N = base.shape[0]
    idx = torch.randperm(N, device=device)[:n_sub]
    nn = torch.cdist(base, base[idx]).argmin(1)
    mix = torch.zeros_like(base)
    for w,f in zip(weights, flats):
        if w == 0: continue
        if f is base: mix += w*base
        else:
            t = sinkhorn_transport(base[idx], f[torch.randperm(f.shape[0],device=device)[:n_sub]])
            mix += w*t[nn]
    return mix.reshape(H,W,C).permute(2,0,1).unsqueeze(0), ref
def decode_style(st, ref):
    L=max(FMS)
    return to_pil(decode_with_style_maps(ref, {s:(st if s==L else F.interpolate(st,size=s,mode='bilinear',align_corners=False)) for s in set(FMS)}))

K_REG=4; TILE=256
rr2 = random.Random(31)
regions = [rr2.choice(sorted(glob.glob(f'{DTD_ROOT}/{rr2.choice(cats)}/*.jpg'))) for _ in range(K_REG)]
print('regions:', [cat_of(p) for p in regions])
comp = Image.new('RGB',(TILE*2,TILE*2))
for i,p in enumerate(regions):
    comp.paste(safe_open(p).resize((TILE,)*2), ((i%2)*TILE,(i//2)*TILE))

fig, ax = plt.subplots(1,5, figsize=(16,3.6))
ax[0].imshow(comp.resize((320,)*2)); ax[0].axis('off'); ax[0].set_title('input: 4 regions', fontsize=10)
for j,eta in enumerate([0.25,0.5,0.75,1.0]):
    out_arr = np.asarray(comp).copy()
    for i,p in enumerate(regions):
        st, ref = bary_style([p]+regions, [1-eta]+[eta/K_REG]*K_REG, anchor=0)
        tile = np.asarray(decode_style(st, ref).resize((TILE,)*2))
        y,x = (i//2)*TILE, (i%2)*TILE
        out_arr[y:y+TILE, x:x+TILE] = tile
    im = Image.fromarray(out_arr); im.save(f'{OUT}/harmonise_eta{int(eta*100):03d}.png')
    ax[j+1].imshow(im.resize((320,)*2)); ax[j+1].axis('off')
    ax[j+1].set_title(f'$\\eta$={eta} toward barycentre', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT}/harmonisation.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# APPLICATION 3 — text painted with barycentric mixtures over a
# continuous 2D palette (Darzi Fig. 16 / Texture Mixer Fig. 1).
# ============================================================
PAL_N, PAL_TILE = 4, 256
CH, CW = 512, 2048
WORD = 'CVPR 2026'

rr3 = random.Random(101)
pref = [c for c in ['honeycombed','cracked','scaly','woven','fibrous','marbled'] if c in cats]
corners = [rr3.choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg'))) for c in pref[:4]]
print('corners:', [cat_of(p) for p in corners])

def bw(u,v): return [(1-u)*(1-v), u*(1-v), (1-u)*v, u*v]
def mixture_at(u,v):
    w = bw(u,v); st, ref = bary_style(corners, w, anchor=int(np.argmax(w)))
    return decode_style(st, ref)

print('rendering palette...')
palette = Image.new('RGB',(PAL_TILE*PAL_N,)*2); pcache={}
for iy in range(PAL_N):
    for ix in range(PAL_N):
        t = mixture_at(ix/(PAL_N-1), iy/(PAL_N-1)).resize((PAL_TILE,)*2)
        pcache[(ix,iy)] = t; palette.paste(t, (ix*PAL_TILE, iy*PAL_TILE))
palette.save(f'{OUT}/palette.png')

def get_font(sz):
    for p in ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']:
        if os.path.exists(p): return ImageFont.truetype(p, sz)
    return ImageFont.load_default()
FONT = get_font(int(CH*0.62))
def layout(word, font):
    pr = ImageDraw.Draw(Image.new('L',(10,10)))
    ws = [pr.textbbox((0,0),ch,font=font)[2]-pr.textbbox((0,0),ch,font=font)[0] for ch in word]
    gap = (CW-sum(ws))/(len(word)+1); xs=[]; x=gap
    for w in ws: xs.append(x); x += w+gap
    bb = pr.textbbox((0,0),word,font=font)
    return xs, (CH-(bb[3]-bb[1]))//2 - bb[1]
def gmask(ch,x,y,f=4):
    m=Image.new('L',(CW,CH),0); ImageDraw.Draw(m).text((x,y),ch,fill=255,font=FONT)
    return np.asarray(m.filter(ImageFilter.GaussianBlur(f))).astype(np.float32)/255.
def field(u,v): return np.asarray(mixture_at(u,v).resize((CW,CH), Image.BICUBIC)).astype(np.float32)

bg = field(0.5,0.9)
bg_f = dino_feats([Image.fromarray(bg.astype(np.uint8)).resize((256,)*2)])[0]
cands = [(u,v) for u in [0.,0.35,0.65,1.] for v in [0.,0.35,0.65,1.]]
scored = sorted(((float(np.linalg.norm(
    dino_feats([pcache[(min(int(u*(PAL_N-1)+.5),PAL_N-1), min(int(v*(PAL_N-1)+.5),PAL_N-1))]])[0]-bg_f)), u, v)
    for u,v in cands), reverse=True)
chars = [c for c in WORD if c != ' ']
uv = [(u,v) for _,u,v in scored[:len(chars)]] if len(scored)>=len(chars) else \
     [(u,v) for _,u,v in scored]*((len(chars)//len(scored))+1)

xs, y = layout(WORD, FONT)
canvas = bg.copy(); ci=0
for ch, x in zip(WORD, xs):
    if ch == ' ': continue
    u,v = uv[ci % len(uv)]; ci += 1
    canvas = canvas*(1-gmask(ch,x,y)[...,None]) + field(u,v)*gmask(ch,x,y)[...,None]
out_img = Image.fromarray(np.clip(canvas,0,255).astype(np.uint8))
out_img.save(f'{OUT}/texture_painting.png')

fig = plt.figure(figsize=(15,9))
gs = fig.add_gridspec(2,1,height_ratios=[1.4,1])
a0=fig.add_subplot(gs[0]); a0.imshow(palette); a0.axis('off')
a0.set_title(f'{PAL_N}x{PAL_N} barycentric palette from four corner textures', fontsize=11)
a1=fig.add_subplot(gs[1]); a1.imshow(out_img); a1.axis('off')
a1.set_title('text painted with palette mixtures, soft-blended into an interpolated background', fontsize=11)
plt.tight_layout(); plt.savefig(f'{OUT}/painting_full.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# CELL 4b — Darzi-style self-loop: x_{t+1} = M(x_t, x_t, 0.5).
# M(A,A) = A is an axiom, so ALL drift is pipeline degradation.
# Also runs the harder variant: x_{t+1} = M(x_t, B, 0.5) with B fixed.
# ============================================================
pairs = pd.read_csv(f'{LOOP_OUT}/pairs.csv')
N_SELF = 8
N_PAIRS_SELF = 12          # Vacher is the bottleneck: ~53s per call
r_dino = manifold(dino_feats, seed=0)
r_inc  = manifold(inc_feats, seed=0)

def ps_dist(a, b):
    return float(np.linalg.norm(ps_stats(a) - ps_stats(b)))

METHODS_SELF = list(FNS) + ['vacher_wasser', 'vacher_gram']
rows = []; t0 = time.time()
for _, row in pairs.iloc[:N_PAIRS_SELF].iterrows():
    k = int(row.k)
    A = Image.open(f'{LOOP_OUT}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
    fA = dino_feats([A])[0]
    for m in METHODS_SELF:
        frames = []
        x = A
        n_it = N_SELF if m in FNS else 4        # Vacher: 4 iterations
        for t in range(n_it):
            if m in FNS:
                x = our_interp(m, x, x, 0.5)
            else:
                qa = prep_square(x, f'{TMPV}/sa.png'); qb = prep_square(x, f'{TMPV}/sb.png')
                x = vacher_interp(qa, qb, 0.5, m.replace('vacher_','tex_'))
            frames.append(x)
        Fm = dino_feats(frames)
        rec = {'pair':k,'method':m,'loop':'self','n':len(frames),
               'realism_0': r_dino(Fm[0]), 'realism_N': r_dino(Fm[-1]),
               'drop': r_dino(Fm[0]) - r_dino(Fm[-1]),
               'drift_from_A': float(np.linalg.norm(Fm[-1]-fA)),
               'drift_per_step': float(np.mean(np.linalg.norm(np.diff(Fm,axis=0),axis=1))),
               'ps_drift': ps_dist(frames[-1], A),
               'traj': json.dumps([r_dino(f) for f in Fm])}
        rows.append(rec)
        if k < 2:
            for t,f in enumerate(frames): f.save(f'{OUT}/self_{m}_{k:03d}_{t:02d}.png')
    print(f'  pair {k} ({(time.time()-t0)/60:.1f}m)')

# TM self-loop needs Phase 1 frames — flag if absent
tm_self = sorted(glob.glob(f'{LOOP_OUT}/tmself_*_*.png'))
if tm_self:
    for _, row in pairs.iloc[:N_PAIRS_SELF].iterrows():
        k = int(row.k)
        fs = sorted(glob.glob(f'{LOOP_OUT}/tmself_{k:03d}_*.png'))
        if not fs: continue
        A = Image.open(f'{LOOP_OUT}/{k:03d}_A.png').convert('RGB').resize((IMG_SIZE,)*2)
        frames = [Image.open(f).convert('RGB').resize((IMG_SIZE,)*2) for f in fs]
        Fm = dino_feats(frames); fA = dino_feats([A])[0]
        rows.append({'pair':k,'method':'texture_mixer','loop':'self','n':len(frames),
                     'realism_0': r_dino(Fm[0]), 'realism_N': r_dino(Fm[-1]),
                     'drop': r_dino(Fm[0]) - r_dino(Fm[-1]),
                     'drift_from_A': float(np.linalg.norm(Fm[-1]-fA)),
                     'drift_per_step': float(np.mean(np.linalg.norm(np.diff(Fm,axis=0),axis=1))),
                     'ps_drift': ps_dist(frames[-1], A),
                     'traj': json.dumps([r_dino(f) for f in Fm])})
else:
    print('\n>>> TM self-loop frames missing — run Cell 2b in the TF notebook')

dfs = pd.DataFrame(rows); dfs.to_csv(f'{OUT}/self_loop.csv', index=False)

print(f"\n=== SELF-LOOP DEGRADATION: x <- M(x,x,0.5), n={N_PAIRS_SELF} pairs ===")
print(f"  M(A,A)=A is an axiom, so ALL drift below is degradation.")
print(f"  {'method':16} {'realism drop':>13} {'drift/step':>11} {'drift from A':>13} {'PS drift':>10}")
for m in dfs.method.unique():
    s = dfs[dfs.method==m]
    print(f"  {m:16} {s['drop'].median():13.4f} {s.drift_per_step.median():11.2f} "
          f"{s.drift_from_A.median():13.2f} {s.ps_drift.median():10.3f}")

print(f"\n=== PAIRED: ours vs each baseline (drift from A; lower better) ===")
for ours in ['ot','ot_sym']:
    for base in [b for b in dfs.method.unique() if b not in ('ot','ot_sym')]:
        a = dfs[dfs.method==ours].set_index('pair').drift_from_A
        b = dfs[dfs.method==base].set_index('pair').drift_from_A
        common = a.index.intersection(b.index)
        if len(common) < 5: continue
        d = a[common] - b[common]
        p = wilcoxon(d).pvalue if d.abs().sum() > 0 else 1.0
        print(f"  {ours:7} vs {base:15} wins {int((d<0).sum()):2d}/{len(d)}  "
              f"median diff {d.median():+.2f}  p={p:.3g}")

fig, ax = plt.subplots(1,2, figsize=(13,4))
for m in dfs.method.unique():
    s = dfs[dfs.method==m]
    tr = np.array([json.loads(t) for t in s.traj if len(json.loads(t))>=4])
    if len(tr): ax[0].plot(range(tr.shape[1]), np.median(tr,axis=0), marker='o', label=m)
ax[0].set_xlabel('self-loop iteration'); ax[0].set_ylabel('DINOv2 realism')
ax[0].set_title('self-loop: does the texture survive its own operator?')
ax[0].grid(alpha=.25); ax[0].legend(fontsize=8)
ms = list(dfs.method.unique())
ax[1].bar(range(len(ms)), [dfs[dfs.method==m].drift_from_A.median() for m in ms])
ax[1].set_xticks(range(len(ms))); ax[1].set_xticklabels(ms, rotation=30, ha='right', fontsize=8)
ax[1].set_ylabel('DINO drift from A after N loops'); ax[1].set_title('total drift (lower = more stable)')
ax[1].grid(alpha=.25, axis='y')
plt.tight_layout(); plt.savefig(f'{OUT}/self_loop.png', dpi=170, bbox_inches='tight'); plt.show()

# filmstrip for the paper
SHOW_M = ['ot_sym','pixel_linear','vacher_wasser']
fig, ax = plt.subplots(len(SHOW_M), N_SELF+1, figsize=(1.7*(N_SELF+1), 1.9*len(SHOW_M)))
for i,m in enumerate(SHOW_M):
    fs = sorted(glob.glob(f'{OUT}/self_{m}_000_*.png'))
    A = Image.open(f'{LOOP_OUT}/000_A.png').convert('RGB')
    ax[i,0].imshow(A); ax[i,0].axis('off')
    ax[i,0].set_title('input' if i==0 else '', fontsize=8)
    ax[i,0].set_ylabel(m, fontsize=8)
    for j in range(N_SELF):
        ax[i,j+1].axis('off')
        if j < len(fs):
            ax[i,j+1].imshow(Image.open(fs[j]))
            if i==0: ax[i,j+1].set_title(f'loop {j+1}', fontsize=8)
plt.suptitle('self-loop degradation: M(x,x,0.5) applied repeatedly', fontsize=11)
plt.tight_layout(); plt.savefig(f'{OUT}/self_loop_strip.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# CELL 8b — CVPR 2026, legible version.
# Corners chosen for luminance contrast; dark background, bright letters.
# ============================================================
def mean_lum(p): return float(np.asarray(p.convert('L'), dtype=np.float32).mean())

cand_cats = [c for c in ['honeycombed','cracked','woven','scaly','fibrous','marbled',
                         'grid','chequered','studded','bumpy'] if c in cats]
pool = []
for c in cand_cats:
    for s in range(4):
        p = random.Random(f'{c}{s}').choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
        pool.append((mean_lum(safe_open(p)), p))
pool.sort()
corners = [pool[0][1], pool[len(pool)//3][1], pool[2*len(pool)//3][1], pool[-1][1]]
print('corners by luminance:', [(round(mean_lum(safe_open(p))), cat_of(p)) for p in corners])

PAL_N, PAL_TILE = 4, 256
CH, CW = 512, 2048
def bw(u,v): return [(1-u)*(1-v), u*(1-v), (1-u)*v, u*v]
def mixture_at(u,v):
    w = bw(u,v); st, ref = bary_style(corners, w, anchor=int(np.argmax(w)))
    return decode_style(st, ref)

print('rendering palette...')
palette = Image.new('RGB',(PAL_TILE*PAL_N,)*2); pc={}
for iy in range(PAL_N):
    for ix in range(PAL_N):
        t = mixture_at(ix/(PAL_N-1), iy/(PAL_N-1)).resize((PAL_TILE,)*2)
        pc[(ix,iy)] = t; palette.paste(t, (ix*PAL_TILE, iy*PAL_TILE))
palette.save(f'{OUT}/palette_v2.png')

# darkest palette cell -> background; brightest -> letters
lums = {k: mean_lum(v) for k,v in pc.items()}
dark = min(lums, key=lums.get); bright = max(lums, key=lums.get)
print(f'background lum {lums[dark]:.0f}, letters lum {lums[bright]:.0f}, '
      f'contrast {lums[bright]-lums[dark]:.0f}')
uv_bg = (dark[0]/(PAL_N-1), dark[1]/(PAL_N-1))
uv_fg = (bright[0]/(PAL_N-1), bright[1]/(PAL_N-1))

def field(u,v): return np.asarray(mixture_at(u,v).resize((CW,CH), Image.BICUBIC)).astype(np.float32)
bg = field(*uv_bg) * 0.75                     # darken further for contrast
fg = np.clip(field(*uv_fg) * 1.15, 0, 255)    # brighten the letters

WORD = 'CVPR 2026'
def get_font(sz):
    for p in ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']:
        if os.path.exists(p): return ImageFont.truetype(p, sz)
    return ImageFont.load_default()
FONT = get_font(int(CH*0.55))
pr = ImageDraw.Draw(Image.new('L',(10,10)))
bb = pr.textbbox((0,0), WORD, font=FONT)
x0 = (CW-(bb[2]-bb[0]))//2 - bb[0]; y0 = (CH-(bb[3]-bb[1]))//2 - bb[1]
m = Image.new('L',(CW,CH),0); ImageDraw.Draw(m).text((x0,y0), WORD, fill=255, font=FONT)
alpha = np.asarray(m.filter(ImageFilter.GaussianBlur(2))).astype(np.float32)/255.
# thin dark outline so letters read even where luminance is close
edge = np.asarray(m.filter(ImageFilter.GaussianBlur(7))).astype(np.float32)/255.
canvas = bg*(1-edge[...,None]*0.5)
canvas = canvas*(1-alpha[...,None]) + fg*alpha[...,None]
out_img = Image.fromarray(np.clip(canvas,0,255).astype(np.uint8))
out_img.save(f'{OUT}/cvpr2026.png')

fig = plt.figure(figsize=(15,9))
gs = fig.add_gridspec(2,1,height_ratios=[1.3,1])
a0=fig.add_subplot(gs[0]); a0.imshow(palette); a0.axis('off')
a0.set_title('barycentric palette: four corner textures, bilinear simplex weights', fontsize=11)
a1=fig.add_subplot(gs[1]); a1.imshow(out_img); a1.axis('off')
a1.set_title('letters and background are different mixtures from the same palette', fontsize=11)
plt.tight_layout(); plt.savefig(f'{OUT}/cvpr2026_full.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# CELL 8c — each letter is a DIFFERENT palette mixture.
# The point is control over local appearance, not legibility.
# ============================================================
# Four visually distinct corners — different structure AND colour
picks = {'honeycombed': 'warm regular cells', 'cracked': 'dark irregular',
         'woven': 'diagonal weave', 'scaly': 'overlapping plates'}
corners = [random.Random(f'v3{c}').choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
           for c in picks if c in cats][:4]
print('corners:', [cat_of(p) for p in corners])

PAL_N, PAL_TILE = 5, 200
def bw(u,v): return [(1-u)*(1-v), u*(1-v), (1-u)*v, u*v]
def mixture_at(u,v):
    w = bw(u,v); st, ref = bary_style(corners, w, anchor=int(np.argmax(w)))
    return decode_style(st, ref)

print('rendering palette...')
palette = Image.new('RGB',(PAL_TILE*PAL_N,)*2); pc={}
for iy in range(PAL_N):
    for ix in range(PAL_N):
        t = mixture_at(ix/(PAL_N-1), iy/(PAL_N-1)).resize((PAL_TILE,)*2)
        pc[(ix,iy)] = t; palette.paste(t, (ix*PAL_TILE, iy*PAL_TILE))

# --- one distinct mixture per letter, spread across the palette ---
CH, CW = 512, 2048
WORD = 'CVPR2026'
uv = [(0,0),(1,0),(0,1),(1,1),(0.5,0),(0,0.5),(1,0.5),(0.5,1)][:len(WORD)]

def get_font(sz):
    for p in ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']:
        if os.path.exists(p): return ImageFont.truetype(p, sz)
    return ImageFont.load_default()
FONT = get_font(int(CH*0.7))

def tile_to(img, H, W):
    o = Image.new('RGB',(W,H))
    for y in range(0,H,img.height):
        for x in range(0,W,img.width): o.paste(img,(x,y))
    return o

pr = ImageDraw.Draw(Image.new('L',(10,10)))
widths = [pr.textbbox((0,0),ch,font=FONT)[2]-pr.textbbox((0,0),ch,font=FONT)[0] for ch in WORD]
gap = (CW-sum(widths))/(len(WORD)+1)
bbw = pr.textbbox((0,0),WORD,font=FONT); y0 = (CH-(bbw[3]-bbw[1]))//2 - bbw[1]

canvas = np.full((CH,CW,3), 245, dtype=np.float32)      # plain light background
x = gap
for ch, w, (u,v) in zip(WORD, widths, uv):
    tex = np.asarray(tile_to(mixture_at(u,v), CH, CW)).astype(np.float32)
    m = Image.new('L',(CW,CH),0); ImageDraw.Draw(m).text((x,y0), ch, fill=255, font=FONT)
    a = np.asarray(m.filter(ImageFilter.GaussianBlur(1.5))).astype(np.float32)[...,None]/255.
    canvas = canvas*(1-a) + tex*a
    x += w + gap
out_img = Image.fromarray(np.clip(canvas,0,255).astype(np.uint8))
out_img.save(f'{OUT}/cvpr2026_v3.png')

fig = plt.figure(figsize=(15,10))
gs = fig.add_gridspec(3,1,height_ratios=[0.6,1.5,1])
a0 = fig.add_subplot(gs[0])
a0.imshow(np.hstack([np.asarray(safe_open(p).resize((200,)*2)) for p in corners])); a0.axis('off')
a0.set_title('four source textures: ' + ', '.join(cat_of(p) for p in corners), fontsize=10)
a1 = fig.add_subplot(gs[1]); a1.imshow(palette); a1.axis('off')
a1.set_title(f'{PAL_N}x{PAL_N} palette — every cell is an OT barycentre with different simplex weights', fontsize=10)
a2 = fig.add_subplot(gs[2]); a2.imshow(out_img); a2.axis('off')
a2.set_title('each character is filled with a different mixture from the palette', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT}/cvpr2026_v3_full.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# Same palette, linear blending — the control that proves synthesis
def mixture_linear(u, v):
    w = bw(u, v)
    sms = []
    for p in corners:
        _, bf, si = get_gen_inputs(cnt_load(safe_open(p)))
        sms.append(build_style_maps(bf, si))
    blob_ref, _, _ = get_gen_inputs(cnt_load(safe_open(corners[int(np.argmax(w))])))
    mixed = {s: sum(wi*sm[s] for wi, sm in zip(w, sms)) for s in set(FMS)}
    return to_pil(decode_with_style_maps(blob_ref, mixed))

pal_lin = Image.new('RGB',(PAL_TILE*PAL_N,)*2)
for iy in range(PAL_N):
    for ix in range(PAL_N):
        pal_lin.paste(mixture_linear(ix/(PAL_N-1), iy/(PAL_N-1)).resize((PAL_TILE,)*2),
                      (ix*PAL_TILE, iy*PAL_TILE))
fig, ax = plt.subplots(1,2, figsize=(13,6.5))
for a,(t,im) in zip(ax, [('linear blending', pal_lin), ('OT barycentre (ours)', palette)]):
    a.imshow(im); a.axis('off'); a.set_title(t, fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT}/palette_ot_vs_linear.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# The maximally ambiguous point: all four weights = 0.25
w_eq = [0.25]*4
st, ref = bary_style(corners, w_eq, anchor=0)
ot_center = decode_style(st, ref)
lin_center = mixture_linear(0.5, 0.5)

# and the 2-way midpoints between opposite corners, where your
# original experiment found the largest gap
pairs_c = [(0,3), (1,2)]
fig, ax = plt.subplots(len(pairs_c)+1, 3, figsize=(9, 3.2*(len(pairs_c)+1)))
ax[0,0].imshow(np.hstack([np.asarray(safe_open(p).resize((120,)*2)) for p in corners]))
ax[0,0].axis('off'); ax[0,0].set_title('four corners', fontsize=9)
ax[0,1].imshow(lin_center); ax[0,1].axis('off'); ax[0,1].set_title('4-way linear (0.25 each)', fontsize=9)
ax[0,2].imshow(ot_center); ax[0,2].axis('off'); ax[0,2].set_title('4-way OT barycentre', fontsize=9)
for r,(i,j) in enumerate(pairs_c):
    A, B = safe_open(corners[i]), safe_open(corners[j])
    d = build_d(cnt_load(A), cnt_load(B))
    ax[r+1,0].imshow(np.hstack([np.asarray(A.resize((120,)*2)), np.asarray(B.resize((120,)*2))]))
    ax[r+1,0].axis('off'); ax[r+1,0].set_title(f'{cat_of(corners[i])} + {cat_of(corners[j])}', fontsize=9)
    ax[r+1,1].imshow(to_pil(interp_lin_pixel(d,0.5))); ax[r+1,1].axis('off')
    ax[r+1,1].set_title('linear midpoint', fontsize=9)
    ax[r+1,2].imshow(to_pil(interp_ot_sym(d,0.5))); ax[r+1,2].axis('off')
    ax[r+1,2].set_title('OT-sym midpoint', fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT}/ot_vs_linear_center.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# BARYCENTRE FIGURE — 2-way (OT and OT-sym) vs linear, plus 4-way
# with three anchoring strategies including a symmetric variant.
# ============================================================

# ---------- k-way barycentre, three anchor strategies ----------
@torch.no_grad()
def _style_flat(p):
    """Return the flattened largest style map and the blob for one texture."""
    img = cnt_load(safe_open(p) if isinstance(p,(str,Path)) else p)
    blob, bf, si = get_gen_inputs(img)
    sm = build_style_maps(bf, si)[max(FMS)][0]
    C,H,W = sm.shape
    return sm.permute(1,2,0).reshape(-1,C), blob, (C,H,W)

@torch.no_grad()
def bary_anchored(paths, weights, anchor, n_sub=4096):
    """k-way OT barycentre expressed on `anchor`'s support (asymmetric)."""
    flats, blobs, shape = [], [], None
    for p in paths:
        f, b, s = _style_flat(p); flats.append(f); blobs.append(b); shape = s
    C,H,W = shape
    base = flats[anchor]; N = base.shape[0]
    idx = torch.randperm(N, device=device)[:n_sub]
    nn  = torch.cdist(base, base[idx]).argmin(1)
    mix = torch.zeros_like(base)
    for w, f in zip(weights, flats):
        if w == 0: continue
        if f is base:
            mix += w * base
        else:
            j = torch.randperm(f.shape[0], device=device)[:n_sub]
            mix += w * sinkhorn_transport(base[idx], f[j])[nn]
    return mix.reshape(H,W,C).permute(2,0,1).unsqueeze(0), blobs[anchor]

@torch.no_grad()
def bary_sym(paths, weights, n_sub=4096):
    """Symmetric k-way barycentre: average the anchored solutions over ALL
    anchor choices, so no input texture is privileged. The k-way analogue
    of OT-sym."""
    outs = []
    for a in range(len(paths)):
        st, ref = bary_anchored(paths, weights, anchor=a, n_sub=n_sub)
        outs.append((st, ref, weights[a]))
    # decode each anchoring and average in image space (supports differ,
    # so feature-space averaging is not well defined across anchors)
    imgs = [np.asarray(decode_style(st, ref), dtype=np.float32) for st, ref, _ in outs]
    return Image.fromarray(np.clip(np.mean(imgs, axis=0), 0, 255).astype(np.uint8))

@torch.no_grad()
def bary_linear(paths, weights):
    """k-way linear blend of style maps — the control."""
    sms = []
    for p in paths:
        _, bf, si = get_gen_inputs(cnt_load(safe_open(p) if isinstance(p,(str,Path)) else p))
        sms.append(build_style_maps(bf, si))
    ref_i = int(np.argmax(weights))
    _, bf, si = get_gen_inputs(cnt_load(safe_open(paths[ref_i])))
    blob = model.encode(cnt_load(safe_open(paths[ref_i])))
    mixed = {s: sum(w*sm[s] for w, sm in zip(weights, sms)) for s in set(FMS)}
    return to_pil(decode_with_style_maps(blob, mixed))

# ---------- FIGURE 1: 2-way, linear vs OT vs OT-sym ----------
PAIR_CATS = [('honeycombed','scaly'), ('cracked','woven'),
             ('marbled','grid'), ('fibrous','chequered')]
pair_paths = []
for a,b in PAIR_CATS:
    if a in cats and b in cats:
        pair_paths.append((random.Random(f'p{a}').choice(sorted(glob.glob(f'{DTD_ROOT}/{a}/*.jpg'))),
                           random.Random(f'p{b}').choice(sorted(glob.glob(f'{DTD_ROOT}/{b}/*.jpg')))))
print('pairs:', [(cat_of(a), cat_of(b)) for a,b in pair_paths])

fig, ax = plt.subplots(len(pair_paths), 5, figsize=(15, 3.1*len(pair_paths)))
ax = np.atleast_2d(ax)
for r,(pa,pb) in enumerate(pair_paths):
    A, B = safe_open(pa), safe_open(pb)
    d = build_d(cnt_load(A), cnt_load(B))
    panels = [(f'A [{cat_of(pa)}]', A.resize((IMG_SIZE,)*2)),
              ('linear', to_pil(interp_lin_pixel(d, 0.5))),
              ('OT', to_pil(interp_ot(d, 0.5))),
              ('OT-sym', to_pil(interp_ot_sym(d, 0.5))),
              (f'B [{cat_of(pb)}]', B.resize((IMG_SIZE,)*2))]
    for c,(t,im) in enumerate(panels):
        ax[r,c].imshow(im); ax[r,c].axis('off')
        if r == 0: ax[r,c].set_title(t, fontsize=11)
        if c in (0,4): ax[r,c].set_title(t, fontsize=9)
plt.suptitle('Two-texture midpoints: transport preserves element structure, averaging blurs it', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_2way.png', dpi=170, bbox_inches='tight'); plt.show()

# ---------- FIGURE 2: 4-way, does symmetrising rescue it? ----------
QUAD = [c for c in ['honeycombed','cracked','woven','scaly'] if c in cats][:4]
quad = [random.Random(f'q{c}').choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg'))) for c in QUAD]
W_EQ = [0.25]*4
print('quad:', [cat_of(p) for p in quad])

lin4 = bary_linear(quad, W_EQ)
ot4_a0 = decode_style(*bary_anchored(quad, W_EQ, anchor=0))
ot4_a2 = decode_style(*bary_anchored(quad, W_EQ, anchor=2))
ot4_sym = bary_sym(quad, W_EQ)

fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
ax[0].imshow(np.hstack([np.asarray(safe_open(p).resize((128,)*2)) for p in quad])); ax[0].axis('off')
ax[0].set_title('four sources', fontsize=10)
for c,(t,im) in enumerate([('4-way linear', lin4),
                           ('OT, anchor=0', ot4_a0),
                           ('OT, anchor=2', ot4_a2),
                           ('OT-sym (all anchors)', ot4_sym)]):
    ax[c+1].imshow(im); ax[c+1].axis('off'); ax[c+1].set_title(t, fontsize=10)
plt.suptitle('Four-texture barycentre at equal weights — anchor dependence and its fix', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_4way.png', dpi=170, bbox_inches='tight'); plt.show()

# quantify the anchor dependence the symmetric version removes
anchored = [decode_style(*bary_anchored(quad, W_EQ, anchor=a)) for a in range(4)]
Fa = dino_feats(anchored)
D = np.linalg.norm(Fa[:,None]-Fa[None,:], axis=2)
src_sep = float(np.mean([np.linalg.norm(x-y) for x,y in
                         zip(dino_feats([safe_open(p).resize((IMG_SIZE,)*2) for p in quad])[:-1],
                             dino_feats([safe_open(p).resize((IMG_SIZE,)*2) for p in quad])[1:])]))
print(f'\nanchor dependence: mean pairwise distance between the four anchored '
      f'barycentres = {D[np.triu_indices(4,1)].mean():.2f}')
print(f'  (source separation for reference: {src_sep:.2f} -> '
      f'{100*D[np.triu_indices(4,1)].mean()/max(src_sep,1e-9):.0f}% of it)')

# ---------- FIGURE 3: palette with OT-sym, as the application ----------
PAL_N, PAL_TILE = 5, 200
def bw(u,v): return [(1-u)*(1-v), u*(1-v), (1-u)*v, u*v]
print('\nrendering OT-sym palette (slower: k anchor solves per cell)...')
pal_sym = Image.new('RGB', (PAL_TILE*PAL_N,)*2); pc_sym = {}
for iy in range(PAL_N):
    for ix in range(PAL_N):
        w = bw(ix/(PAL_N-1), iy/(PAL_N-1))
        t = bary_sym(quad, w).resize((PAL_TILE,)*2)
        pc_sym[(ix,iy)] = t; pal_sym.paste(t, (ix*PAL_TILE, iy*PAL_TILE))
    print(f'  row {iy+1}/{PAL_N}')
pal_sym.save(f'{OUT}/palette_otsym.png')

plt.figure(figsize=(7.5,7.5)); plt.imshow(pal_sym); plt.axis('off')
plt.title('barycentric palette (OT-sym): every cell is a synthesised texture,\n'
          'none appear in the dataset', fontsize=11)
plt.tight_layout(); plt.savefig(f'{OUT}/palette_otsym_fig.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# HEADLINE FIGURE — k-way anchor dependence and its fix.
# Two rows of source quadruples; for each: all k anchored barycentres
# side by side (the defect) plus the symmetric solution (the fix),
# with the anchor-dependence measured in DINOv2 space.
# ============================================================
QUADS = [
    ['honeycombed', 'cracked', 'woven', 'scaly'],
    ['marbled', 'grid', 'fibrous', 'bumpy'],
]
W_EQ = [0.25] * 4
TILE = 256

def quad_paths(cs, tag):
    return [random.Random(f'{tag}{c}').choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
            for c in cs if c in cats][:4]

rows = []
for qi, qc in enumerate(QUADS):
    q = quad_paths(qc, f'hero{qi}')
    if len(q) < 4:
        print(f'skipping quad {qi}: only {len(q)} categories available'); continue
    anchored = [decode_style(*bary_anchored(q, W_EQ, anchor=a)) for a in range(4)]
    sym      = bary_sym(q, W_EQ)
    lin      = bary_linear(q, W_EQ)

    Fa = dino_feats(anchored)
    D  = np.linalg.norm(Fa[:, None] - Fa[None, :], axis=2)
    spread = float(D[np.triu_indices(4, 1)].mean())
    Fs = dino_feats([safe_open(p).resize((IMG_SIZE,)*2) for p in q])
    Ds = np.linalg.norm(Fs[:, None] - Fs[None, :], axis=2)
    src_sep = float(Ds[np.triu_indices(4, 1)].mean())

    # symmetric control: is the sym solution stable across re-runs?
    sym2 = bary_sym(q, W_EQ)
    sym_noise = float(np.linalg.norm(dino_feats([sym])[0] - dino_feats([sym2])[0]))

    rows.append({'paths': q, 'anchored': anchored, 'sym': sym, 'lin': lin,
                 'spread': spread, 'src_sep': src_sep, 'sym_noise': sym_noise})
    print(f'quad {qi} [{", ".join(cat_of(p) for p in q)}]')
    print(f'  anchor spread {spread:.1f}  |  source separation {src_sep:.1f}  '
          f'|  ratio {100*spread/max(src_sep,1e-9):.0f}%  |  sym re-run noise {sym_noise:.1f}')

# ---------------- figure ----------------
NCOL = 7   # sources | 4 anchored | sym | linear
fig, ax = plt.subplots(len(rows), NCOL, figsize=(2.35*NCOL, 2.75*len(rows)))
ax = np.atleast_2d(ax)
for r, rec in enumerate(rows):
    strip = np.hstack([np.asarray(safe_open(p).resize((TILE//2,)*2)) for p in rec['paths']])
    ax[r,0].imshow(strip); ax[r,0].axis('off')
    if r == 0: ax[r,0].set_title('four sources', fontsize=11, pad=8)
    for a in range(4):
        ax[r,a+1].imshow(rec['anchored'][a]); ax[r,a+1].axis('off')
        if r == 0: ax[r,a+1].set_title(f'anchor = {a+1}', fontsize=11, pad=8, color='#B85042')
    ax[r,5].imshow(rec['sym']); ax[r,5].axis('off')
    if r == 0: ax[r,5].set_title('OT-sym\n(all anchors)', fontsize=11, pad=4)
    ax[r,6].imshow(rec['lin']); ax[r,6].axis('off')
    if r == 0: ax[r,6].set_title('linear blend', fontsize=11, pad=8)
    ax[r,1].text(-0.06, 0.5, f"anchor spread\n{100*rec['spread']/rec['src_sep']:.0f}% of |sources|",
                 transform=ax[r,1].transAxes, ha='right', va='center', fontsize=9, color='#B85042')

plt.suptitle('Four textures, equal weights (0.25 each): the barycentre depends on which source anchors it',
             fontsize=13, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f'{OUT}/hero_kway_anchor.png', dpi=200, bbox_inches='tight')
plt.show()

print('\n=== HEADLINE NUMBERS ===')
for i, rec in enumerate(rows):
    print(f"  quad {i}: anchor dependence = {100*rec['spread']/rec['src_sep']:.0f}% of source separation, "
          f"vs {100*rec['sym_noise']/rec['src_sep']:.0f}% re-run noise for OT-sym")
print(f"\nsaved: {OUT}/hero_kway_anchor.png")

In [ ]:
# Is anchor dependence real, or within the operator's own noise?
# Compare across-anchor spread vs. same-anchor re-run spread, per quad.
for qi, qc in enumerate(QUADS):
    q = quad_paths(qc, f'hero{qi}')
    if len(q) < 4: continue
    across = dino_feats([decode_style(*bary_anchored(q, W_EQ, anchor=a)) for a in range(4)])
    within = dino_feats([decode_style(*bary_anchored(q, W_EQ, anchor=0)) for _ in range(4)])
    Da = np.linalg.norm(across[:,None]-across[None,:], axis=2)[np.triu_indices(4,1)]
    Dw = np.linalg.norm(within[:,None]-within[None,:], axis=2)[np.triu_indices(4,1)]
    print(f'quad {qi}: across-anchor {Da.mean():.1f} | same-anchor re-runs {Dw.mean():.1f} '
          f'| ratio {Da.mean()/max(Dw.mean(),1e-9):.2f}x')